In [0]:
%pip install  pycountry
import pycountry
import gettext
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

# 1. Configurar traducción automática al español desde los archivos locales de la librería
try:
    es_active = gettext.translation('iso3166', pycountry.LOCALES_DIR, languages=['es'])
except FileNotFoundError:
    es_active = gettext.translation('iso3166-1', pycountry.LOCALES_DIR, languages=['es'])

try:
    es_historic = gettext.translation('iso3166-3', pycountry.LOCALES_DIR, languages=['es'])
except FileNotFoundError:
    es_historic = None

# 2. Función UDF inteligente sin harcodeos
def codigo_a_espanol_dinamico(texto_pais):
    if not texto_pais:
        return None
    
    texto_limpio = str(texto_pais).strip()
    
    # --- PASO 1: Buscar en países activos (Soporta 'USA', 'Spain', 'Colombia', 'UK' por iniciales) ---
    try:
        # Intenta búsqueda difusa (Fuzzy Search) que es la más tolerante a formatos variados
        resultados = pycountry.countries.search_fuzzy(texto_limpio)
        if resultados:
            pais_activo = resultados[0]
            return es_active.gettext(pais_activo.name)
    except LookupError:
        pass  # Si no lo encuentra en activos, continúa al bloque histórico
    
    # --- PASO 2: Buscar en países históricos (Soporta 'USSR') ---
    try:
        # El módulo historic_countries requiere buscar en los atributos directamente
        for pais_hist in pycountry.historic_countries:
            # Compara contra el nombre o variaciones de códigos del registro histórico
            if texto_limpio.upper() in [getattr(pais_hist, attr, "").upper() for attr in ['name', 'alpha_2', 'alpha_3', 'alpha_4']]:
                if es_historic:
                    return es_historic.gettext(pais_hist.name)
                return pais_hist.name
    except Exception:
        pass

    return "Desconocido"

# 3. Registrar la UDF en PySpark
convertir_dinamico_udf = F.udf(codigo_a_espanol_dinamico, StringType())


from pyspark.sql import DataFrame
from pyspark.sql import functions as F

from pyspark.sql import functions as F
from pyspark.sql.types import StringType

# 1. Crear el diccionario offline en memoria (Rápido y local)
f1_diccionario = {
    # Europa
    "British": "Británico", "French": "Francés", "German": "Alemán",
    "Italian": "Italiano", "Dutch": "Neerlandés", "Spanish": "Español",
    "Belgian": "Belga", "Swiss": "Suizo", "Austrian": "Austriaco",
    "Monaco": "Monegasco", "Monegasque": "Monegasco", "Liechtenstein": "Liechtensteiniano",
    "Finnish": "Finlandés", "Swedish": "Sueco", "Danish": "Danés", 
    "Norwegian": "Noruego", "Russian": "Ruso", "Polish": "Polaco", 
    "Czech": "Checo", "Hungarian": "Húngaro", "Romanian": "Rumano",
    # Norte América
    "American": "Estadounidense", "Canadian": "Canadiense", "Mexican": "Mexicano",
    "Argentinian": "Argentino", "Argentine": "Argentino", "Brazilian": "Brasileño", 
    "Chilean": "Chileno", "Colombian": "Colombiano", "Ecuadorian": "Ecuatoriano",
    "Peruvian": "Peruano", "Uruguayan": "Uruguayo", "Venezuelan": "Venezolano",
    # Asia
    "Japanese": "Japonés", "Chinese": "Chino", "Indian": "Indio", 
    "Thai": "Tailandés", "Malaysian": "Malayo", "Indonesian": "Indonesio",
    # Oceania
    "Australian": "Australiano", "New Zealander": "Neozelandés", "Kiwi": "Neozelandés",
    # Africa
    "South African": "Sudafricano", "Moroccan": "Marroquí", "Rhodesian": "Rodesiano",
    # Naciones desaparecidas
    "West German": "Alemán Occidental", "East German": "Alemán Oriental"
}

# 2. Definir la función lógica de Python
def traducir_con_diccionario(valor_ingles):
    if not valor_ingles:
        return None
    
    # Limpiar espacios y estandarizar formato Título (ej: "  dutch " -> "Dutch")
    clave = str(valor_ingles).strip().title()
    
    # Si la palabra está en el diccionario, la traduce; si no, deja la original
    return f1_diccionario.get(clave, valor_ingles)

# 3. Registrar la función como UDF en PySpark
traducir_f1_udf = F.udf(traducir_con_diccionario, StringType())
